# Machine learning and fund characteristics: selecting European equity funds with positive alpha

This notebook replicates the design of DeMiguel, Gil-Bazo, Nogales and Santos (2023, *Journal of Financial Economics*), "Machine learning and fund characteristics help to select mutual funds with positive alpha", on actively managed European equity funds (Morningstar data).

Each year, models are trained to predict a fund's realized six-factor alpha from its lagged characteristics. The funds with the highest predicted alpha are bought and held for twelve months. The resulting portfolios are evaluated out of sample with factor models in euros.

The reusable code lives in `src/fundml/`. This notebook runs the pipeline in order and shows the intermediate diagnostics. Settings (file names, filters, windows, predictors) are in `src/fundml/config.py`.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fundml import config as C
from fundml import data, alphas, panel, experiment, evaluation, plots
from fundml import factors as ff

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
plots.set_style()
TABLES = C.OUTPUT_DIR / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
print(f"data: {C.DATA_DIR} | unit: {C.UNIT} | quick mode: {C.QUICK}")

## 1. Morningstar data

The Morningstar exports are wide (one column per month or year). They are reshaped to long tables keyed by share class (`SecId`) and month:

- Monthly returns: converted from % to decimals.
- Monthly net assets.
- Monthly Morningstar category.
- Annual net expense ratios: converted from % p.a. to a monthly decimal rate (÷ 1200), so they are directly comparable with monthly returns.

Every table is checked for uniqueness on its key before any merge.

In [ ]:
raw = data.load_morningstar()
long = data.reshape_morningstar(raw)
selected_categories = pd.read_excel(C.DATA_DIR / C.FILES["selected_categories"])["category"].tolist()

ret = long["returns"]
obs = ret.groupby("SecId")["monthly_return"].count()
print(f"share classes: {ret['SecId'].nunique():,} | months: {ret['date'].nunique()} | "
      f"return NA rate: {ret['monthly_return'].isna().mean():.2%}")
obs.describe(percentiles=[0.25, 0.5, 0.75]).rename("return observations per share class")

## 2. Monthly panel

`data.build_monthly_panel` performs the following steps:

1. **Returns × net assets** (inner join). The one-month lag of net assets is kept only when the previous record is exactly one calendar month earlier, so gaps never produce stale lags.
2. **Expense ratio** of the calendar year. Non-positive values are set to missing.
3. **Net flows**: $(TNA_t - TNA_{t-1}(1+r_t))/TNA_{t-1}$, computed only if lagged net assets are at least EUR 100,000, to avoid exploding ratios from tiny denominators. Flows are winsorised at 1%/99% within each month.
4. **Characteristics and category**: static characteristics and the time-varying Morningstar category are added. Index products are removed with a name-based screen (index, ETF, MSCI, tracker, …). Share-class age is computed from the inception date.
5. **Universe**: EEA-focused investment areas, the preselected Morningstar categories, and equity allocation above 70%.
6. **Turnover**: converted from % to a fraction. Values above 1000% p.a. or negative are treated as invalid, and the rest is winsorised by month.

The size/age entry screen is applied later (section 6). Rolling betas may use the full return history, including months before a fund enters the sample.

*Caveat:* equity allocation, turnover and manager tenure exist only as a single Morningstar snapshot taken at the download date. See README, "Limitations".

In [ ]:
monthly_sc = data.build_monthly_panel(raw, long, selected_categories)
monthly_sc["net_assets_l1"].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.99])

## 3. Unit of analysis

Share classes of the same fund hold the same portfolio. Their returns, alphas and factor exposures are nearly identical; only fees, size and flows differ. Treating them as separate observations has three consequences:

- The top decile is filled with copies of the same funds.
- The effective sample size is inflated.
- Cross-validation leaks information when siblings land in both the training and the validation fold.

By default (`C.UNIT = "fund"`), share classes are therefore aggregated to the fund level (`FundId`):

- Returns, expense ratio and turnover are weighted by lagged class net assets.
- Net assets are summed, and age is the oldest class's age.
- Flows are recomputed from fund net assets.

Setting `FUNDML_UNIT=share_class` reproduces the share-class design of the thesis.

In [ ]:
if C.UNIT == "fund":
    monthly = data.aggregate_to_fund_level(monthly_sc)
else:
    monthly = monthly_sc.copy()
print(f"{C.UNIT} panel: {len(monthly):,} rows, {monthly['secid'].nunique():,} units")

## 4. Factors in EUR

Kenneth French's European factors are denominated in USD. For a EUR investor they are converted as follows:

- **Long-short factors**: $LS^{EUR} = LS^{USD}/(1+r^{FX})$ (Glück et al.), where $r^{FX}$ is the monthly change in USD per EUR.
- **Market**: the gross USD market return is converted to EUR, and the EUR risk-free rate is subtracted.

The **EUR risk-free rate** is the lower of the 1-month EURIBOR and an overnight-index proxy. Both are compounded monthly with ACT/360. The proxy is EONIA until December 2021 and €STR afterwards.

**Timing check:** converting European equity returns from USD back to EUR removes currency noise, so the EUR market factor must be *less* volatile than the USD one. A one-month misalignment of the FX series would make it more volatile.

In [ ]:
fac = ff.build_factor_panel()
print(f"factor panel: {fac.index.min():%Y-%m} to {fac.index.max():%Y-%m}, {len(fac)} months")
display_cols = ["r_fx_usd_per_eur", "rf_usd", "rf_eur", "mkt-rf_usd", "mkt-rf_eur"]
print(ff.fx_alignment_check(fac))
fac[display_cols].describe().T

## 5. Rolling realized alphas

For each fund and month $m$, factor loadings are estimated by OLS on the 36 calendar months $[m-36, m-1]$; at least 30 observations are required. The realized alpha is

$$\alpha_{i,m} = r^{e}_{i,m} - \hat\beta_{i,m}'\, f_m,$$

where the intercept is not subtracted (DeMiguel et al., 2023). This is computed for the CAPM (1F), Fama-French 3F, Carhart 4F and FF5 + momentum (6F) models.

From the 6F window regression, the $R^2$ and the HAC t-statistics (3 lags) of the intercept and of each loading are kept as fund characteristics.

Alphas are estimated twice, once with the EUR-converted and once with the raw USD factors.

In [ ]:
eur_cols = [c for c in fac.columns if c.endswith("_eur")]
usd_cols = [c for c in fac.columns if c.endswith("_usd")]
alphas_eur = alphas.rolling_alpha(monthly.merge(fac[eur_cols], left_on="date", right_index=True, how="left"), "eur")
alphas_usd = alphas.rolling_alpha(monthly.merge(fac[usd_cols], left_on="date", right_index=True, how="left"), "usd")
alphas_eur[alphas.ALPHA_OUTPUT_COLS].describe().T

## 6. Entry screen and estimation sample

To limit incubation bias, a fund enters the sample from the first month in which it has reached net assets of at least EUR 5 million **and** an age of at least 36 months. Rolling-regression outputs (estimated on the full history in section 5) are then attached to the screened panel.

Monthly value added (Berk and van Binsbergen) is computed as gross alpha times lagged size: $(\alpha^{6F} + ER)\times TNA_{t-1}$, in EUR million.

In [ ]:
sample = data.apply_entry_screen(monthly)
sample = panel.attach_alphas(sample, alphas_eur)

desc_vars = ["monthly_return", "Monthly6Alpha", "net_flow_winsor", "value_added", "expense_ratio",
             "class_age_months", "turnover_ratio_frac_w", "alpha_tstat", "Monthly6Factor_R2",
             "manager_tenure", "net_assets_mln"]
desc = pd.DataFrame({"N": sample[desc_vars].count(), "mean": sample[desc_vars].mean(),
                     "median": sample[desc_vars].median(), "sd": sample[desc_vars].std()})
desc.to_csv(TABLES / "descriptive_statistics.csv")
desc.round(4)

## 7. EUR-adjusted vs USD factors

This compares fund alphas under EUR-converted factors with alphas under unconverted USD factors, on the estimation sample.

Inference uses the monthly cross-sectional mean of the difference (one observation per month) with Newey-West standard errors (12 lags). Fund-month observations within a month share the same factor and currency shocks. An observation-level paired t-test therefore treats hundreds of thousands of highly correlated observations as independent and overstates significance.

In [ ]:
keys = sample[["secid", "date"]]
currency_cmp = alphas.compare_currencies(alphas_eur.merge(keys), alphas_usd.merge(keys))
currency_cmp.to_csv(TABLES / "eur_vs_usd_alpha.csv")
print("mean 6F R^2:", {k: round(v, 3) for k, v in currency_cmp.attrs["mean_R2_6F"].items()})
currency_cmp.round(4).T

## 8. Annual panel and machine-learning panel

**Annual panel.** For each fund-year:

- Alpha, value added and flows are annualised (12 × monthly mean).
- Flow volatility is the standard deviation of monthly flows × √12.
- Year-end characteristics are taken from December.

**ML panel.** Row $(i, t)$ contains the characteristics of year $t-1$, standardised cross-sectionally within $t-1$ (missing values set to 0, the cross-sectional mean), and the realized alpha of year $t$. Specifically:

- Characteristics exist only if the fund has at least 10 monthly alphas in $t-1$. Lags therefore never skip a missing year, and no fund enters with an all-zero predictor vector.
- The year-$t$ universe is every fund with characteristics at the end of $t-1$. Funds that disappear during $t$ stay in the universe with a missing target. They were investable on 1 January, so dropping them would condition selection on future survival.
- Manager tenure and turnover are snapshot variables (see section 2) and are excluded from the main predictor set (`C.PREDICTORS`). A robustness run adds them back.

In [ ]:
annual = panel.build_annual_panel(sample)
ml = panel.build_ml_panel(annual)
PREDICTORS = panel.predictor_columns(C.PREDICTORS)
ALL_PREDICTORS = panel.predictor_columns(C.PREDICTOR_BASE)

print(f"ML panel: {len(ml):,} fund-years, {ml['year'].min()}-{ml['year'].max()}, "
      f"{len(PREDICTORS)} predictors; target missing (fund left during year): "
      f"{ml[C.TARGET].isna().mean():.1%}")
ml.groupby("year").agg(funds=("secid", "size"), with_target=(C.TARGET, "count")).T

In [ ]:
plots.correlation_matrix(ml, ALL_PREDICTORS)
plt.show()

## 9. Out-of-sample fund selection

The experiment uses an expanding window. The first 10 years are the initial training sample. Each subsequent year $t$:

1. Every model is trained on all fund-years before $t$.
2. Hyperparameters are tuned with **time-ordered cross-validation**: each of the last five training years is used once as the validation year, with the model trained on all earlier years. The tuning criterion is the Spearman information coefficient (IC), because the task is ranking funds.
3. Alpha is predicted for every fund in the year-$t$ universe.
4. The top 20% / 10% of funds by predicted alpha are bought and held for 12 months. Two weightings are used: prediction-weighted (RW) and equally weighted (EW). If a fund disappears during the year, its weight is reinvested pro rata in the remaining holdings.

The models are:

- OLS.
- Ridge, Lasso, Bayesian ridge and elastic net.
- Gradient-boosted stumps (XGBoost, or scikit-learn if XGBoost is not installed) and LightGBM, if installed.
- Shallow random forests.
- An **ensemble**: the equal-weight average of all non-OLS models. Its membership is fixed in advance, not chosen by performance.

Turnover is the sum of absolute differences between the new weights and the previous portfolio's drifted end-of-year weights.

In [ ]:
main = experiment.run_oos_experiment(ml, sample, PREDICTORS, scoring="ic", cv="expanding")

## 10. Portfolio performance

All portfolio returns are net of fees and evaluated in EUR.

- **Alphas**: from the FF3 + momentum, FF5, and FF5 + momentum models, with Newey-West standard errors (12 lags).
- **Alpha relative to OLS**: each ML strategy minus the OLS top-decile EW portfolio.
- **Risk measures**: Sharpe ratio, Sortino ratio, information ratio, maximum drawdown, historical 99% VaR, and turnover.
- **Signal quality**: the out-of-sample Spearman IC per model.

In [ ]:
ev_main = evaluation.evaluate(main, fac)
for k in ["alpha", "alpha_vs_ols", "risk", "ic"]:
    if ev_main[k] is not None:
        ev_main[k].to_csv(TABLES / f"main_{k}.csv")

## 11. Passive benchmarks

Both benchmarks hold the whole year-$t$ universe, with weights set at formation:

- **EW**: equal weights.
- **AW**: weights proportional to net assets at the end of $t-1$.

In [ ]:
bench = experiment.run_benchmarks(ml, sample)
bench_alpha = evaluation.alpha_table(bench, fac)
bench_risk = evaluation.risk_table(bench, fac)
bench_alpha.to_csv(TABLES / "benchmarks_alpha.csv")
print(evaluation.format_alpha_table(bench_alpha).to_string())
bench_risk.round(3)

## 12. Which characteristics matter?

These are the coefficients (linear models) and impurity importances (tree models) of the models that were actually used in each test year. They are averaged across years together with:

- their dispersion,
- the share of years with a positive sign,
- the selection frequency, i.e. the share of years with a non-zero coefficient (relevant for Lasso and elastic net).

In [ ]:
imp = evaluation.importance_summary(main)
imp.to_csv(TABLES / "feature_importance.csv", index=False)
plots.feature_importance(imp)
plt.show()
imp[imp["model"].isin(["ridge", "enet", "rf"])].round(4).head(30)

## 13. Capital allocation across predicted-alpha deciles

Each test year, funds are sorted into deciles by the ensemble's predicted alpha. Size and fees are measured **at formation** (end of $t-1$). Measuring size at the end of the holding year would mix in the flows that respond to that year's performance.

If the highest-alpha deciles are not the largest funds, capital has not (yet) flowed to the most skilled managers (Berk and Green, 2004).

In [ ]:
deciles = evaluation.capital_misallocation(main, ml, model="ensemble")
deciles.to_csv(TABLES / "capital_misallocation.csv")
plots.misallocation(deciles)
plt.show()
deciles.round(3)

## 14. Robustness

Each specification changes one element of the main design:

| Specification | Change |
|---|---|
| `kfold_ic` | Shuffled 5-fold CV (thesis design) instead of time-ordered CV |
| `expanding_rmse` | Tuning by RMSE instead of IC |
| `kfold_rmse` | Shuffled CV and RMSE |
| `gross` | Portfolios evaluated on gross returns (net + expense ratio) |
| `with_snapshot_predictors` | Manager tenure and turnover added back as predictors |

In [ ]:
specs = {
    "kfold_ic": dict(scoring="ic", cv="kfold"),
    "expanding_rmse": dict(scoring="rmse", cv="expanding"),
    "kfold_rmse": dict(scoring="rmse", cv="kfold"),
}
robust = {"main": main}
for name, kw in specs.items():
    print(f"--- {name}")
    robust[name] = experiment.run_oos_experiment(ml, sample, PREDICTORS, verbose=False, **kw)

sample_gross = panel.add_gross_returns(sample)
print("--- gross")
robust["gross"] = experiment.run_oos_experiment(ml, sample_gross, PREDICTORS,
                                                ret_col="gross_return", verbose=False)
print("--- with_snapshot_predictors")
robust["with_snapshot_predictors"] = experiment.run_oos_experiment(ml, sample, ALL_PREDICTORS,
                                                                   verbose=False)

In [ ]:
robust_summary = evaluation.summary_across_specs(robust, fac)
robust_summary.to_csv(TABLES / "robustness_summary.csv")
robust_summary.round(3)

## 15. Cumulative alpha

Monthly realized alpha of each portfolio is $r^e_t - \hat\beta' f_t$, with $\hat\beta$ estimated over the full out-of-sample period (DeMiguel et al., 2023, footnote 27). It is cumulated over time. The comparison covers the top-decile prediction-weighted portfolios and the two passive benchmarks.

In [ ]:
rets_main, _ = evaluation.combine_rw_ew(main)
show = ["ols_top_10_RW", "ridge_top_10_RW", "xgb_top_10_RW", "rf_top_10_RW", "ensemble_top_10_RW"]
series = pd.concat([rets_main[[c for c in show if c in rets_main]], bench], axis=1)
for spec_name, cols in C.FACTOR_MODELS.items():
    cum = evaluation.cumulative_alpha(series, fac, cols)
    plots.cumulative_alpha(cum, f"Cumulative alpha, top decile vs benchmarks ({spec_name})",
                           f"cumulative_alpha_{spec_name}.png")
    plt.show()